In [6]:
import torch
import torch.nn as nn 
import os
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPProcessor
import faiss
import numpy as np
import pickle
import h5py

In [7]:
class KNNFeatureAugmenter:
    def __init__(self, k=10, embedding_dim=512, use_gpu=False):
        """
        KNN-based feature augmentation for embeddings.
        
        Args:
            k: Number of nearest neighbors to consider
            embedding_dim: Dimension of the embeddings
            use_gpu: Whether to use GPU for FAISS operations (set to False for CPU-only)
        """
        self.k = k
        self.embedding_dim = embedding_dim
        self.use_gpu = use_gpu
        
        # Initialize FAISS indices for image and text embeddings
        self.img_index = None
        self.text_index = None
        
        # Store embeddings for meta-feature calculation (on CPU for FAISS)
        self.img_embeddings_cpu = None
        self.text_embeddings_cpu = None
        
        # Keep GPU versions for fast computation
        self.img_embeddings_gpu = None
        self.text_embeddings_gpu = None
        
    def _create_faiss_index(self, embeddings_cpu):
        """Create FAISS index for embeddings (CPU-only)."""
        # Ensure embeddings are on CPU and float32 for FAISS
        embeddings_np = embeddings_cpu.cpu().numpy().astype(np.float32)
        
        # Create CPU index (Inner product for cosine similarity with normalized vectors)
        index = faiss.IndexFlatIP(self.embedding_dim)
        
        # Add embeddings to index
        index.add(embeddings_np)
        
        return index
    
    def fit(self, img_embeddings, text_embeddings):
        """
        Build KNN indices from training embeddings.
        
        Args:
            img_embeddings: Image embeddings tensor [N, embedding_dim]
            text_embeddings: Text embeddings tensor [N, embedding_dim]
        """
        # Normalize embeddings for cosine similarity
        img_embeddings_norm = torch.nn.functional.normalize(img_embeddings, p=2, dim=1)
        text_embeddings_norm = torch.nn.functional.normalize(text_embeddings, p=2, dim=1)
        
        # Store CPU versions for FAISS
        self.img_embeddings_cpu = img_embeddings_norm.cpu()
        self.text_embeddings_cpu = text_embeddings_norm.cpu()
        
        # Store GPU versions for fast computation (if available)
        if img_embeddings.is_cuda:
            self.img_embeddings_gpu = img_embeddings_norm
            self.text_embeddings_gpu = text_embeddings_norm
        else:
            self.img_embeddings_gpu = self.img_embeddings_cpu
            self.text_embeddings_gpu = self.text_embeddings_cpu
        
        # Create FAISS indices (CPU-only)
        self.img_index = self._create_faiss_index(self.img_embeddings_cpu)
        self.text_index = self._create_faiss_index(self.text_embeddings_cpu)
        
    def _calculate_knn_features(self, query_embeddings, index, stored_embeddings_gpu):
        """
        Calculate KNN-based meta-features for query embeddings.
        
        Returns:
            knn_features: Tensor of meta-features [batch_size, 4]
                - mean_similarity: Average similarity to k nearest neighbors
                - std_similarity: Standard deviation of similarities
                - min_similarity: Minimum similarity to k nearest neighbors
                - diversity_score: Measure of diversity among neighbors
        """
        # Move query to CPU for FAISS search
        query_np = query_embeddings.cpu().numpy().astype(np.float32)
        
        # Search for k+1 neighbors (including self)
        similarities, indices = index.search(query_np, self.k + 1)
        
        # Remove self-similarity (first result is usually the query itself)
        similarities = similarities[:, 1:]  # [batch_size, k]
        indices = indices[:, 1:]
        
        # Convert similarities back to GPU tensors
        similarities = torch.tensor(similarities, device=query_embeddings.device, dtype=query_embeddings.dtype)
        
        # Calculate meta-features on GPU
        mean_similarity = similarities.mean(dim=1)
        std_similarity = similarities.std(dim=1)
        min_similarity = similarities.min(dim=1)[0]
        
        # Calculate diversity score (average pairwise distance among neighbors)
        diversity_scores = []
        for i in range(len(indices)):
            neighbor_indices = indices[i]
            # Get neighbor embeddings (use GPU version if available)
            neighbor_embeddings = stored_embeddings_gpu[neighbor_indices]  # [k, embedding_dim]
            
            # Calculate pairwise cosine distances on GPU
            if len(neighbor_embeddings) > 1:
                similarity_matrix = torch.mm(neighbor_embeddings, neighbor_embeddings.t())
                # Get upper triangular part (excluding diagonal)
                triu_indices = torch.triu_indices(len(neighbor_embeddings), len(neighbor_embeddings), offset=1, device=neighbor_embeddings.device)
                pairwise_similarities = similarity_matrix[triu_indices[0], triu_indices[1]]
                diversity_score = 1.0 - pairwise_similarities.mean()  # Higher diversity = lower similarity
            else:
                diversity_score = torch.tensor(0.0, device=query_embeddings.device, dtype=query_embeddings.dtype)
            
            diversity_scores.append(diversity_score)
        
        diversity_scores = torch.stack(diversity_scores)
        
        # Stack all features
        knn_features = torch.stack([
            mean_similarity,
            std_similarity,
            min_similarity,
            diversity_scores
        ], dim=1)
        
        return knn_features
    
    def transform(self, img_embeddings, text_embeddings):
        """
        Calculate KNN meta-features for given embeddings.
        
        Args:
            img_embeddings: Image embeddings tensor [batch_size, embedding_dim]
            text_embeddings: Text embeddings tensor [batch_size, embedding_dim]
            
        Returns:
            img_knn_features: Image KNN meta-features [batch_size, 4]
            text_knn_features: Text KNN meta-features [batch_size, 4]
        """
        if self.img_index is None or self.text_index is None:
            raise ValueError("Must call fit() before transform()")
        
        # Normalize query embeddings
        img_embeddings_norm = torch.nn.functional.normalize(img_embeddings, p=2, dim=1)
        text_embeddings_norm = torch.nn.functional.normalize(text_embeddings, p=2, dim=1)
        
        # Calculate KNN features (CPU search, GPU computation)
        img_knn_features = self._calculate_knn_features(
            img_embeddings_norm, self.img_index, self.img_embeddings_gpu
        )
        text_knn_features = self._calculate_knn_features(
            text_embeddings_norm, self.text_index, self.text_embeddings_gpu
        )
        
        return img_knn_features, text_knn_features

In [8]:
class CLIPPricePredictor(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32", k_neighbors=10):
        super().__init__()
        # Load CLIP model
        self.clip = CLIPModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="cuda"
        )
        
        # Freeze CLIP parameters
        for param in self.clip.parameters():
            param.requires_grad = False
        
        # Get embedding dimension from CLIP
        self.embedding_dim = self.clip.config.projection_dim
        
        # Initialize KNN augmenter (CPU-only for FAISS)
        self.knn_augmenter = KNNFeatureAugmenter(
            k=k_neighbors, 
            embedding_dim=self.embedding_dim, 
            use_gpu=False  # Set to False for CPU-only FAISS
        )
        
        # Flag to track if KNN is fitted
        self.knn_fitted = False
    
    @torch.no_grad()
    def extract_features(self, pixel_values, input_ids, attention_mask):
        """
        Extract frozen multimodal embeddings from CLIP.
        Combines image and text features in the shared embedding space.
        """
        self.clip.eval()
        
        # Get CLIP outputs
        outputs = self.clip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Extract normalized embeddings from CLIP's projection space
        image_embeds = outputs.image_embeds  # [batch_size, projection_dim]
        text_embeds = outputs.text_embeds    # [batch_size, projection_dim]
        
        return image_embeds, text_embeds
    
    def fit_knn(self, dataloader):
        """
        Fit KNN indices using all training data.
        
        Args:
            dataloader: DataLoader containing training samples
        """
        print("Fitting KNN indices...")
        all_img_embeds = []
        all_text_embeds = []
        
        self.eval()
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Extracting embeddings for KNN"):
                pixel_values = batch['pixel_values'].to(next(self.parameters()).device)
                input_ids = batch['input_ids'].to(next(self.parameters()).device)
                attention_mask = batch['attention_mask'].to(next(self.parameters()).device)
                
                img_embeds, text_embeds = self.extract_features(
                    pixel_values, input_ids, attention_mask
                )
                
                # Keep on GPU for now, will be handled in KNN augmenter
                all_img_embeds.append(img_embeds)
                all_text_embeds.append(text_embeds)
        
        # Concatenate all embeddings (keep on GPU)
        all_img_embeds = torch.cat(all_img_embeds, dim=0)
        all_text_embeds = torch.cat(all_text_embeds, dim=0)
        
        # Fit KNN augmenter (it will handle GPU-to-CPU movement internally)
        self.knn_augmenter.fit(all_img_embeds, all_text_embeds)
        self.knn_fitted = True
        print("KNN indices fitted successfully!")
    
    def forward(self, pixel_values, input_ids, attention_mask=None):
        """
        Forward pass that returns all four numerical features.
        
        Returns:
            dict with keys: 'text_embeds', 'img_embeds', 'text_knn', 'img_knn'
        """
        # Extract CLIP features
        with torch.no_grad():
            img_embeds, text_embeds = self.extract_features(
                pixel_values, input_ids, attention_mask
            )
        
        # Calculate KNN features if fitted
        if self.knn_fitted:
            img_knn_features, text_knn_features = self.knn_augmenter.transform(
                img_embeds, text_embeds
            )
        else:
            # Use zeros as placeholder if KNN not fitted
            batch_size = img_embeds.size(0)
            device = img_embeds.device
            img_knn_features = torch.zeros(batch_size, 4, device=device, dtype=img_embeds.dtype)
            text_knn_features = torch.zeros(batch_size, 4, device=device, dtype=text_embeds.dtype)
        
        return {
            'text_embeds': text_embeds,
            'img_embeds': img_embeds, 
            'text_knn': text_knn_features,
            'img_knn': img_knn_features
        }

In [9]:
def extract_and_save_features(model, dataloader, save_path, file_format='h5'):
    print(f"Extracting features and saving to {save_path}...")
    
    all_text_embeds = []
    all_img_embeds = []
    all_text_knn = []
    all_img_knn = []
    all_indices = []
    
    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc="Extracting features")):
            pixel_values = batch['pixel_values'].to(next(model.parameters()).device)
            input_ids = batch['input_ids'].to(next(model.parameters()).device)
            attention_mask = batch['attention_mask'].to(next(model.parameters()).device)
            
            # Get all features
            features = model(pixel_values, input_ids, attention_mask)
            
            # Convert to CPU and collect
            all_text_embeds.append(features['text_embeds'].cpu().numpy())
            all_img_embeds.append(features['img_embeds'].cpu().numpy())
            all_text_knn.append(features['text_knn'].cpu().numpy())
            all_img_knn.append(features['img_knn'].cpu().numpy())
            
            # Track batch indices for reference
            batch_size = len(batch['input_ids'])
            start_idx = batch_idx * dataloader.batch_size
            batch_indices = list(range(start_idx, start_idx + batch_size))
            all_indices.extend(batch_indices)
    
    # Concatenate all features
    text_embeds = np.concatenate(all_text_embeds, axis=0)
    img_embeds = np.concatenate(all_img_embeds, axis=0)
    text_knn = np.concatenate(all_text_knn, axis=0)
    img_knn = np.concatenate(all_img_knn, axis=0)
    indices = np.array(all_indices)
    
    print(f"Feature shapes:")
    print(f"  text_embeds: {text_embeds.shape}")
    print(f"  img_embeds: {img_embeds.shape}")
    print(f"  text_knn: {text_knn.shape}")
    print(f"  img_knn: {img_knn.shape}")
    
    # Save based on format
    if file_format == 'h5':
        with h5py.File(save_path, 'w') as f:
            f.create_dataset('text_embeds', data=text_embeds, compression='gzip')
            f.create_dataset('img_embeds', data=img_embeds, compression='gzip')
            f.create_dataset('text_knn', data=text_knn, compression='gzip')
            f.create_dataset('img_knn', data=img_knn, compression='gzip')
            f.create_dataset('indices', data=indices, compression='gzip')
            
            # Add metadata
            f.attrs['num_samples'] = len(indices)
            f.attrs['text_embed_dim'] = text_embeds.shape[1]
            f.attrs['img_embed_dim'] = img_embeds.shape[1]
            f.attrs['knn_features'] = 4
            
    elif file_format == 'npz':
        np.savez_compressed(
            save_path,
            text_embeds=text_embeds,
            img_embeds=img_embeds,
            text_knn=text_knn,
            img_knn=img_knn,
            indices=indices
        )
        
    elif file_format == 'pkl':
        features_dict = {
            'text_embeds': text_embeds,
            'img_embeds': img_embeds,
            'text_knn': text_knn,
            'img_knn': img_knn,
            'indices': indices
        }
        with open(save_path, 'wb') as f:
            pickle.dump(features_dict, f)
    
    else:
        raise ValueError(f"Unsupported file format: {file_format}")
    
    print(f"Features saved successfully to {save_path}")
    return {
        'text_embeds': text_embeds,
        'img_embeds': img_embeds,
        'text_knn': text_knn,
        'img_knn': img_knn,
        'indices': indices
    }

def load_features(file_path, file_format='h5'):
    if file_format == 'h5':
        features = {}
        with h5py.File(file_path, 'r') as f:
            features['text_embeds'] = f['text_embeds'][:]
            features['img_embeds'] = f['img_embeds'][:]
            features['text_knn'] = f['text_knn'][:]
            features['img_knn'] = f['img_knn'][:]
            features['indices'] = f['indices'][:]
            
    elif file_format == 'npz':
        data = np.load(file_path)
        features = {
            'text_embeds': data['text_embeds'],
            'img_embeds': data['img_embeds'],
            'text_knn': data['text_knn'],
            'img_knn': data['img_knn'],
            'indices': data['indices']
        }
        
    elif file_format == 'pkl':
        with open(file_path, 'rb') as f:
            features = pickle.load(f)
    
    else:
        raise ValueError(f"Unsupported file format: {file_format}")
    
    return features

In [10]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)

In [11]:
# Create test dataset class
class TestDataset(Dataset):
    def __init__(self, csv_file, image_dir, processor):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.processor = processor
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sample_id = row['sample_id']
        
        # Get image filename from image_link
        image_link = row['image_link']
        image_filename = image_link.split('/')[-1]  # Extract filename from URL
        image_path = os.path.join(self.image_dir, image_filename)
        
        try:
            # Load image
            image = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            # Create a black image as fallback
            image = Image.new('RGB', (224, 224), color='black')
        
        # Concatenate text fields
        text_fields = []
        if pd.notna(row['Item Name']):
            text_fields.append(str(row['Item Name']))
        if pd.notna(row['Product Description']):
            text_fields.append(str(row['Product Description']))
        if pd.notna(row['Bullet Points']):
            text_fields.append(str(row['Bullet Points']))
        
        text = " ".join(text_fields) if text_fields else "No description available"
        
        return {
            'sample_id': sample_id,
            'image': image,
            'text': text
        }

In [ ]:
def load_model():
    # Initialize model
    model = CLIPPricePredictor(k_neighbors=10)
    
    # Load trained weights
    model_path = "/home/arnavw/Documents/amazon-ml-2025/src/model_final_2.pth"
    checkpoint = torch.load(model_path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load only the regressor weights (CLIP weights are frozen)
    if 'regressor' in checkpoint:
        model.regressor.load_state_dict(checkpoint['regressor'])
        print("Loaded regressor weights from checkpoint")
    elif 'model_state_dict' in checkpoint:
        # Try to load full model state dict
        model.load_state_dict(checkpoint['model_state_dict'], strict=False)
        print("Loaded model weights from checkpoint")
    else:
        # Assume checkpoint contains the full state dict
        model.load_state_dict(checkpoint, strict=False)
        print("Loaded weights directly from checkpoint")
    
    model.eval()
    return model

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model = load_model().to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


Using device: cuda
Loaded regressor weights from checkpoint


In [6]:
# Custom collate function to handle variable sized texts
def custom_collate_fn(batch):
    sample_ids = [item['sample_id'] for item in batch]
    images = [item['image'] for item in batch]
    texts = [item['text'] for item in batch]
    
    # Process all images and texts together for consistent padding
    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    )
    
    return {
        'sample_id': sample_ids,
        'pixel_values': inputs['pixel_values'],
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask']
    }


In [ ]:
# Setup paths
csv_path = "/home/arnavw/Documents/amazon-ml-2025/dataset/test_split/part1.csv"
image_dir = "/home/arnavw/Documents/amazon-ml-2025/images/test_part1"
output_path = "/home/arnavw/Documents/amazon-ml-2025/test_predictions_1.csv"


# Create dataset and dataloader
test_dataset = TestDataset(csv_path, image_dir, processor)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0, collate_fn=custom_collate_fn)

print(f"Total samples to process: {len(test_dataset)}")
print(f"Number of batches: {len(test_dataloader)}")

Total samples to process: 25000
Number of batches: 3125


In [8]:
# Run inference
def run_inference(t_dataloader):
    predictions = []
    sample_ids = []
    
    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(t_dataloader, desc="Running inference")):
            try:
                # Move to device
                pixel_values = batch['pixel_values'].to(device)
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                
                # Forward pass
                outputs = model(pixel_values, input_ids, attention_mask)
                
                # Store predictions
                predictions.extend(outputs.cpu().numpy().tolist())
                sample_ids.extend(batch['sample_id'])
                
            except Exception as e:
                print(f"Error processing batch {batch_idx}: {e}")
                # Add zeros for failed batches
                batch_size = len(batch['sample_id'])
                predictions.extend([0.0] * batch_size)
                sample_ids.extend(batch['sample_id'])
    
    return sample_ids, predictions

In [ ]:
# Run inference
print("Starting inference...")
sample_ids, predictions = run_inference()
print(f"Inference completed. Processed {len(sample_ids)} samples.")

In [16]:
# Save predictions to CSV
results_df = pd.DataFrame({
    'sample_id': sample_ids,
    'predicted_price': predictions
})

# Sort by sample_id to ensure consistent ordering
results_df = results_df.sort_values('sample_id')
results_df['predicted_price'] = np.expm1(results_df['predicted_price']).round(2)
# Save to CSV
results_df.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(results_df.head(10))

print(f"\nSummary statistics:")
print(f"Total predictions: {len(results_df)}")
print(f"Mean predicted price: ${results_df['predicted_price'].mean():.2f}")
print(f"Min predicted price: ${results_df['predicted_price'].min():.2f}")
print(f"Max predicted price: ${results_df['predicted_price'].max():.2f}")

Predictions saved to /home/arnavw/Documents/amazon-ml-2025/test_predictions.csv

First 10 predictions:
       sample_id  predicted_price
4020           1            12.42
6852           9            43.79
10141         20            23.55
7442          28             8.02
5757          61            18.33
24766         86            29.60
1001         100            25.69
10138        102            19.68
7034         127             7.58
15321        134            18.08

Summary statistics:
Total predictions: 25000
Mean predicted price: $17.22
Min predicted price: $1.34
Max predicted price: $204.99


#### Part 2 

In [9]:
del sample_ids, predictions
import gc 
gc.collect()
torch.cuda.empty_cache()

NameError: name 'sample_ids' is not defined

In [10]:
csv_path = "/home/arnavw/Documents/amazon-ml-2025/dataset/test_split/part2.csv"
image_dir = "/home/arnavw/Documents/amazon-ml-2025/images/test_part2"
output_path = "/home/arnavw/Documents/amazon-ml-2025/test_predictions_2.csv"

test_dataset = TestDataset(csv_path, image_dir, processor)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=custom_collate_fn)

print(f"Total samples to process: {len(test_dataset)}")
print(f"Number of batches: {len(test_dataloader)}")

Total samples to process: 25000
Number of batches: 3125


In [11]:
# Run inference
print("Starting inference...")
sample_ids, predictions = run_inference(test_dataloader)
print(f"Inference completed. Processed {len(sample_ids)} samples.")

Starting inference...


Running inference:  68%|██████▊   | 2130/3125 [13:37<05:22,  3.08it/s]

Error loading image /home/arnavw/Documents/amazon-ml-2025/images/test_part2/813CjSgHj0S.jpg: [Errno 2] No such file or directory: '/home/arnavw/Documents/amazon-ml-2025/images/test_part2/813CjSgHj0S.jpg'


Running inference: 100%|██████████| 3125/3125 [19:35<00:00,  2.66it/s]

Inference completed. Processed 25000 samples.


In [12]:
# Save predictions to CSV
results_df = pd.DataFrame({
    'sample_id': sample_ids,
    'predicted_price': predictions
})

# Sort by sample_id to ensure consistent ordering
results_df = results_df.sort_values('sample_id')
results_df['predicted_price'] = np.expm1(results_df['predicted_price']).round(2)
# Save to CSV
results_df.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(results_df.head(10))

print(f"\nSummary statistics:")
print(f"Total predictions: {len(results_df)}")
print(f"Mean predicted price: ${results_df['predicted_price'].mean():.2f}")
print(f"Min predicted price: ${results_df['predicted_price'].min():.2f}")
print(f"Max predicted price: ${results_df['predicted_price'].max():.2f}")

Predictions saved to /home/arnavw/Documents/amazon-ml-2025/test_predictions_2.csv

First 10 predictions:
       sample_id  predicted_price
14491          3            24.64
13023         21             9.65
3586          23            25.40
7136          29            19.55
13710         48            11.71
18453         55            13.69
6736          57            13.86
13466         58            10.09
8104          62             9.26
7863          63            19.81

Summary statistics:
Total predictions: 25000
Mean predicted price: $17.21
Min predicted price: $1.29
Max predicted price: $198.41


#### Part 3

In [13]:
del sample_ids, predictions
import gc 
gc.collect()
torch.cuda.empty_cache()

In [14]:
csv_path = "/home/arnavw/Documents/amazon-ml-2025/dataset/test_split/part3.csv"
image_dir = "/home/arnavw/Documents/amazon-ml-2025/images/test_part3"
output_path = "/home/arnavw/Documents/amazon-ml-2025/test_predictions_3.csv"

test_dataset = TestDataset(csv_path, image_dir, processor)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=custom_collate_fn)

print(f"Total samples to process: {len(test_dataset)}")
print(f"Number of batches: {len(test_dataloader)}")

Total samples to process: 25000
Number of batches: 3125


In [15]:
# Run inference
print("Starting inference...")
sample_ids, predictions = run_inference(test_dataloader)
print(f"Inference completed. Processed {len(sample_ids)} samples.")

Starting inference...


Running inference:  42%|████▏     | 1310/3125 [07:27<10:05,  3.00it/s]

Error loading image /home/arnavw/Documents/amazon-ml-2025/images/test_part3/41XNmOHw6iL.jpg: cannot identify image file '/home/arnavw/Documents/amazon-ml-2025/images/test_part3/41XNmOHw6iL.jpg'


Running inference: 100%|██████████| 3125/3125 [17:56<00:00,  2.90it/s]

Inference completed. Processed 25000 samples.


In [16]:
# Save predictions to CSV
results_df = pd.DataFrame({
    'sample_id': sample_ids,
    'predicted_price': predictions
})

# Sort by sample_id to ensure consistent ordering
results_df = results_df.sort_values('sample_id')
results_df['predicted_price'] = np.expm1(results_df['predicted_price']).round(2)
# Save to CSV
results_df.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(results_df.head(10))

print(f"\nSummary statistics:")
print(f"Total predictions: {len(results_df)}")
print(f"Mean predicted price: ${results_df['predicted_price'].mean():.2f}")
print(f"Min predicted price: ${results_df['predicted_price'].min():.2f}")
print(f"Max predicted price: ${results_df['predicted_price'].max():.2f}")

Predictions saved to /home/arnavw/Documents/amazon-ml-2025/test_predictions_3.csv

First 10 predictions:
       sample_id  predicted_price
14597         19            17.54
5653          25            13.43
10198         27            15.80
24885         45            31.28
24023         50            17.36
15306         67             6.43
18443         71            16.12
18380         82             9.80
8310          99            21.10
18462        116            11.38

Summary statistics:
Total predictions: 25000
Mean predicted price: $17.16
Min predicted price: $1.34
Max predicted price: $209.13


### Finally Put all in one CSV File

In [ ]:
test_df = pd.read_csv("/home/arnavw/Documents/amazon-ml-2025/dataset/test.csv")

# Read all prediction files
pred1_df = pd.read_csv("/home/arnavw/Documents/amazon-ml-2025/test_predictions_1.csv")
pred2_df = pd.read_csv("/home/arnavw/Documents/amazon-ml-2025/test_predictions_2.csv")
pred3_df = pd.read_csv("/home/arnavw/Documents/amazon-ml-2025/test_predictions_3.csv")

# Combine all predictions
all_predictions = pd.concat([pred1_df, pred2_df, pred3_df], ignore_index=True)

# Merge with original test.csv to maintain the same order
final_predictions = test_df[['sample_id']].merge(
    all_predictions, 
    on='sample_id', 
    how='left'
)

final_predictions['sample_id'] = final_predictions['sample_id'].astype(int)
final_predictions['price'] = final_predictions['predicted_price'].astype(float)
final_predictions = final_predictions.drop(columns=['predicted_price'])

# Save the final predictions
final_output_path = "/home/arnavw/Documents/amazon-ml-2025/final_test_predictions.csv"
final_predictions.to_csv(final_output_path, index=False)

print(f"Final predictions saved to {final_output_path}")
print(f"Total samples: {len(final_predictions)}")
print(f"Samples with predictions: {final_predictions['price'].notna().sum()}")
print(f"Missing predictions: {final_predictions['price'].isna().sum()}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(final_predictions.head(10))

# Display summary statistics
valid_predictions = final_predictions['price'].dropna()
if len(valid_predictions) > 0:
    print(f"\nSummary statistics:")
    print(f"Mean predicted price: ${valid_predictions.mean():.2f}")
    print(f"Min predicted price: ${valid_predictions.min():.2f}")
    print(f"Max predicted price: ${valid_predictions.max():.2f}")

Final predictions saved to /home/arnavw/Documents/amazon-ml-2025/final_test_predictions.csv
Total samples: 75000
Samples with predictions: 75000
Missing predictions: 0

First 10 predictions:
   sample_id  price
0     100179  12.64
1     245611  18.83
2     146263  18.29
3      95658   8.27
4      36806  19.05
5     148239   4.45
6      92659  11.16
7       3780  10.15
8     196940  19.92
9      20472   5.88

Summary statistics:
Mean predicted price: $17.20
Min predicted price: $1.29
Max predicted price: $209.13
